In [2]:
#!/usr/bin/env python3
"""
Erosion Feasibility Study — Elastoplastic Simulations
======================================================
Run cells in a notebook or as a script to understand erosion patterns.
"""

import torch
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pathlib import Path
from collections import defaultdict

# ============================================================
# CELL 1: Load simulations
# ============================================================
TEST_DIR = Path("/scratch/jtb3sud/processed_elasto_plastic/global_max/normalized/small/test")
# Also check train set for more data
TRAIN_DIR = Path("/scratch/jtb3sud/processed_elasto_plastic/global_max/normalized/small/train")

EROSION_THRESHOLD = 0.5

def load_sims(data_dir, max_files=None):
    paths = sorted(data_dir.glob("*.pt"))
    if max_files:
        paths = paths[:max_files]
    sims = []
    for p in paths:
        try:
            sim = torch.load(p, weights_only=False)
            sims.append((p.stem, sim))
        except:
            pass
    return sims

# Load all test sims
test_sims = load_sims(TEST_DIR)
print(f"Loaded {len(test_sims)} test simulations")

# Optionally load some train sims too
train_sims = load_sims(TRAIN_DIR, max_files=20)
print(f"Loaded {len(train_sims)} train simulations")

all_sims = test_sims + train_sims

Loaded 8 test simulations
Loaded 20 train simulations


In [3]:
# ============================================================
# CELL 2: Basic erosion statistics
# ============================================================
print("\n" + "=" * 70)
print("EROSION STATISTICS ACROSS ALL SIMULATIONS")
print("=" * 70)

stats = []
for sim_name, sim in all_sims:
    num_elements = sim[0].elements.shape[0] if hasattr(sim[0], 'elements') else 0
    num_timesteps = len(sim)
    
    erosion_per_step = []
    has_erosion = False
    first_erosion_step = None
    
    for t, data in enumerate(sim):
        if hasattr(data, 'x_element') and data.x_element is not None:
            x_elem = data.x_element.cpu().numpy().flatten()
            n_eroded = (x_elem < EROSION_THRESHOLD).sum()
            erosion_per_step.append(n_eroded)
            if n_eroded > 0 and first_erosion_step is None:
                first_erosion_step = t
                has_erosion = True
        else:
            erosion_per_step.append(0)
    
    max_eroded = max(erosion_per_step)
    total_element_timesteps = num_elements * num_timesteps
    total_eroded_element_timesteps = sum(erosion_per_step)
    
    stats.append({
        'name': sim_name,
        'num_elements': num_elements,
        'num_timesteps': num_timesteps,
        'has_erosion': has_erosion,
        'first_erosion_step': first_erosion_step,
        'max_eroded': max_eroded,
        'max_eroded_pct': 100 * max_eroded / max(num_elements, 1),
        'erosion_per_step': erosion_per_step,
        'total_eroded_pct': 100 * total_eroded_element_timesteps / max(total_element_timesteps, 1),
    })

# Summary
n_with_erosion = sum(1 for s in stats if s['has_erosion'])
print(f"\nSimulations with erosion: {n_with_erosion}/{len(stats)} "
      f"({100*n_with_erosion/len(stats):.0f}%)")

if n_with_erosion > 0:
    eroding_stats = [s for s in stats if s['has_erosion']]
    print(f"\nAmong simulations WITH erosion:")
    print(f"  First erosion step:  min={min(s['first_erosion_step'] for s in eroding_stats)}, "
          f"max={max(s['first_erosion_step'] for s in eroding_stats)}, "
          f"mean={np.mean([s['first_erosion_step'] for s in eroding_stats]):.1f}")
    print(f"  Max eroded elements: min={min(s['max_eroded'] for s in eroding_stats)}, "
          f"max={max(s['max_eroded'] for s in eroding_stats)}, "
          f"mean={np.mean([s['max_eroded'] for s in eroding_stats]):.1f}")
    print(f"  Max eroded %:        min={min(s['max_eroded_pct'] for s in eroding_stats):.2f}%, "
          f"max={max(s['max_eroded_pct'] for s in eroding_stats):.2f}%, "
          f"mean={np.mean([s['max_eroded_pct'] for s in eroding_stats]):.2f}%")
    
    # When does erosion start relative to simulation length?
    relative_starts = [s['first_erosion_step'] / s['num_timesteps'] for s in eroding_stats]
    print(f"  Erosion starts at:   {np.mean(relative_starts)*100:.1f}% through simulation (mean)")

# Per-simulation table
print(f"\n{'Simulation':<25} {'Elements':>8} {'Steps':>6} {'Erosion?':>8} {'1st Step':>8} "
      f"{'Max Eroded':>10} {'Max %':>8} {'Total %':>8}")
print("-" * 95)
for s in stats:
    print(f"{s['name']:<25} {s['num_elements']:>8} {s['num_timesteps']:>6} "
          f"{'YES' if s['has_erosion'] else 'no':>8} "
          f"{s['first_erosion_step'] if s['first_erosion_step'] is not None else '-':>8} "
          f"{s['max_eroded']:>10} {s['max_eroded_pct']:>7.2f}% {s['total_eroded_pct']:>7.3f}%")



EROSION STATISTICS ACROSS ALL SIMULATIONS

Simulations with erosion: 24/28 (86%)

Among simulations WITH erosion:
  First erosion step:  min=18, max=38, mean=26.4
  Max eroded elements: min=27, max=163, mean=105.3
  Max eroded %:        min=0.06%, max=0.34%, mean=0.21%
  Erosion starts at:   65.9% through simulation (mean)

Simulation                Elements  Steps Erosion? 1st Step Max Eroded    Max %  Total %
-----------------------------------------------------------------------------------------------
simulation_179               48403     40      YES       25         60    0.12%   0.033%
simulation_191               54012     40      YES       27        107    0.20%   0.036%
simulation_337               50040     40      YES       22        130    0.26%   0.071%
simulation_384               47694     40      YES       35         67    0.14%   0.010%
simulation_395               51050     40      YES       29        109    0.21%   0.036%
simulation_490               49215     40  

In [4]:
# ============================================================
# CELL 3: Erosion temporal patterns
# ============================================================
print("\n" + "=" * 70)
print("EROSION TEMPORAL PATTERNS")
print("=" * 70)

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Plot 1: Erosion count over time for all sims with erosion
ax = axes[0, 0]
for s in stats:
    if s['has_erosion']:
        ax.plot(s['erosion_per_step'], alpha=0.6, label=s['name'][:15])
ax.set_xlabel('Timestep')
ax.set_ylabel('Eroded Elements')
ax.set_title('Erosion Count Over Time (per simulation)')
ax.grid(alpha=0.3)
if n_with_erosion <= 10:
    ax.legend(fontsize=7)

# Plot 2: Erosion rate (new erosions per step)
ax = axes[0, 1]
for s in stats:
    if s['has_erosion']:
        eps = s['erosion_per_step']
        rate = [max(0, eps[t] - eps[t-1]) for t in range(1, len(eps))]
        ax.plot(range(1, len(eps)), rate, alpha=0.6)
ax.set_xlabel('Timestep')
ax.set_ylabel('New Erosions Per Step')
ax.set_title('Erosion Rate (new erosions per timestep)')
ax.grid(alpha=0.3)

# Plot 3: Histogram of first erosion timestep
ax = axes[1, 0]
if n_with_erosion > 0:
    first_steps = [s['first_erosion_step'] for s in stats if s['has_erosion']]
    ax.hist(first_steps, bins=20, edgecolor='black', alpha=0.7, color='coral')
    ax.axvline(np.mean(first_steps), color='red', ls='--', label=f'Mean: {np.mean(first_steps):.0f}')
    ax.legend()
ax.set_xlabel('Timestep')
ax.set_ylabel('Count')
ax.set_title('Distribution: When Does Erosion Start?')
ax.grid(alpha=0.3)

# Plot 4: Max eroded % histogram
ax = axes[1, 1]
if n_with_erosion > 0:
    max_pcts = [s['max_eroded_pct'] for s in stats if s['has_erosion']]
    ax.hist(max_pcts, bins=20, edgecolor='black', alpha=0.7, color='steelblue')
    ax.axvline(np.mean(max_pcts), color='red', ls='--', label=f'Mean: {np.mean(max_pcts):.1f}%')
    ax.legend()
ax.set_xlabel('Max Eroded %')
ax.set_ylabel('Count')
ax.set_title('Distribution: How Much Gets Eroded?')
ax.grid(alpha=0.3)

plt.suptitle('Erosion Feasibility Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('erosion_temporal_patterns.png', dpi=150, bbox_inches='tight')
plt.close()
print("Saved: erosion_temporal_patterns.png")


EROSION TEMPORAL PATTERNS
Saved: erosion_temporal_patterns.png


In [5]:
# ============================================================
# CELL 4: Examine x_element values — is it truly binary?
# ============================================================
print("\n" + "=" * 70)
print("x_element VALUE DISTRIBUTION — IS IT TRULY BINARY?")
print("=" * 70)

all_x_elem_values = []
for sim_name, sim in all_sims[:10]:  # Sample
    for t, data in enumerate(sim):
        if hasattr(data, 'x_element') and data.x_element is not None:
            vals = data.x_element.cpu().numpy().flatten()
            all_x_elem_values.extend(vals)

all_x_elem_values = np.array(all_x_elem_values)
print(f"Total element-timestep samples: {len(all_x_elem_values):,}")
print(f"Unique values: {np.unique(all_x_elem_values)}")
print(f"Min: {all_x_elem_values.min():.6f}")
print(f"Max: {all_x_elem_values.max():.6f}")
print(f"Mean: {all_x_elem_values.mean():.6f}")

# Distribution of non-zero, non-one values
intermediate = all_x_elem_values[(all_x_elem_values > 0.01) & (all_x_elem_values < 0.99)]
print(f"\nValues between 0.01 and 0.99: {len(intermediate):,} "
      f"({100*len(intermediate)/len(all_x_elem_values):.4f}%)")

if len(intermediate) > 0:
    print(f"  These intermediate values: {np.unique(intermediate)[:20]}")
    print("  → There IS a gradual signal! Could model as continuous damage.")
else:
    print("  → Purely binary. No intermediate states available.")

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(all_x_elem_values, bins=100, edgecolor='black', alpha=0.7, log=True)
ax.set_xlabel('x_element value')
ax.set_ylabel('Count (log)')
ax.set_title('Distribution of x_element Values')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('x_element_distribution.png', dpi=150, bbox_inches='tight')
plt.close()
print("Saved: x_element_distribution.png")


x_element VALUE DISTRIBUTION — IS IT TRULY BINARY?
Total element-timestep samples: 20,026,440
Unique values: [0. 1.]
Min: 0.000000
Max: 1.000000
Mean: 0.999552

Values between 0.01 and 0.99: 0 (0.0000%)
  → Purely binary. No intermediate states available.
Saved: x_element_distribution.png


In [6]:
# ============================================================
# CELL 5: Spatial analysis — WHERE does erosion happen?
# ============================================================
print("\n" + "=" * 70)
print("SPATIAL EROSION ANALYSIS")
print("=" * 70)

# For a few sims with erosion, look at which elements erode
# and what the displacement field looks like there
eroding_sims = [(name, sim) for name, sim in all_sims 
                if any(hasattr(d, 'x_element') and d.x_element is not None 
                       and (d.x_element.cpu().numpy().flatten() < EROSION_THRESHOLD).any()
                       for d in sim)]

if eroding_sims:
    fig, axes = plt.subplots(2, min(3, len(eroding_sims)), 
                              figsize=(6*min(3, len(eroding_sims)), 10))
    if len(eroding_sims) == 1:
        axes = axes.reshape(-1, 1)
    
    for col, (sim_name, sim) in enumerate(eroding_sims[:3]):
        elements = sim[0].elements.cpu().numpy()
        pos = sim[0].pos.cpu().numpy() if hasattr(sim[0], 'pos') else sim[0].x[:, :2].cpu().numpy()
        
        # Find first and last timestep with erosion
        for t, data in enumerate(sim):
            if hasattr(data, 'x_element') and data.x_element is not None:
                x_elem = data.x_element.cpu().numpy().flatten()
                if (x_elem < EROSION_THRESHOLD).any():
                    first_erosion_t = t
                    break
        
        last_data = sim[-1]
        last_x_elem = last_data.x_element.cpu().numpy().flatten() if hasattr(last_data, 'x_element') else np.ones(len(elements))
        eroded_mask_final = last_x_elem < EROSION_THRESHOLD
        
        # Get displacement at that timestep
        disp = last_data.y.cpu().numpy() if hasattr(last_data, 'y') else np.zeros((pos.shape[0], 2))
        disp_mag = np.sqrt(disp[:, 0]**2 + disp[:, 1]**2)
        
        # Element centroids
        elem_centroids = pos[elements].mean(axis=1)
        
        # Plot 1: Erosion map (which elements eroded)
        ax = axes[0, col] if axes.ndim > 1 else axes[0]
        ax.scatter(elem_centroids[~eroded_mask_final, 0], 
                  elem_centroids[~eroded_mask_final, 1], 
                  s=2, c='steelblue', alpha=0.3, label='Active')
        if eroded_mask_final.sum() > 0:
            ax.scatter(elem_centroids[eroded_mask_final, 0],
                      elem_centroids[eroded_mask_final, 1],
                      s=20, c='red', marker='x', label='Eroded')
        ax.set_title(f'{sim_name}\nErosion Map (t=final)\n{eroded_mask_final.sum()} eroded', fontsize=9)
        ax.set_aspect('equal')
        ax.legend(fontsize=7)
        ax.grid(alpha=0.2)
        
        # Plot 2: Displacement magnitude at nodes near eroded elements
        ax = axes[1, col] if axes.ndim > 1 else axes[1]
        sc = ax.scatter(pos[:, 0], pos[:, 1], c=disp_mag, s=2, cmap='hot', alpha=0.5)
        if eroded_mask_final.sum() > 0:
            # Highlight nodes of eroded elements
            eroded_nodes = np.unique(elements[eroded_mask_final].flatten())
            ax.scatter(pos[eroded_nodes, 0], pos[eroded_nodes, 1], 
                      s=15, facecolors='none', edgecolors='lime', linewidth=1)
        ax.set_title(f'Displacement + Eroded Nodes (green circles)', fontsize=9)
        ax.set_aspect('equal')
        plt.colorbar(sc, ax=ax, shrink=0.7)
        ax.grid(alpha=0.2)
    
    plt.suptitle('Spatial Erosion Analysis', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig('erosion_spatial_analysis.png', dpi=150, bbox_inches='tight')
    plt.close()
    print("Saved: erosion_spatial_analysis.png")


SPATIAL EROSION ANALYSIS
Saved: erosion_spatial_analysis.png


In [7]:
# ============================================================
# CELL 6: Displacement stats at eroded vs non-eroded elements
# ============================================================
print("\n" + "=" * 70)
print("DISPLACEMENT AT ERODED vs NON-ERODED ELEMENTS")
print("=" * 70)

eroded_disps = []
active_disps = []

for sim_name, sim in all_sims:
    elements = sim[0].elements.cpu().numpy()
    
    for t, data in enumerate(sim):
        if not (hasattr(data, 'x_element') and data.x_element is not None):
            continue
        x_elem = data.x_element.cpu().numpy().flatten()
        eroded = x_elem < EROSION_THRESHOLD
        
        if not eroded.any():
            continue
        
        # Get displacement (y = target for this step)
        disp = data.y.cpu().numpy() if hasattr(data, 'y') else data.x[:, 2:4].cpu().numpy()
        disp_mag = np.sqrt(disp[:, 0]**2 + disp[:, 1]**2)
        
        # Average displacement per element
        elem_disp = disp_mag[elements].mean(axis=1)
        
        eroded_disps.extend(elem_disp[eroded].tolist())
        active_disps.extend(elem_disp[~eroded].tolist())

if eroded_disps:
    eroded_disps = np.array(eroded_disps)
    active_disps = np.array(active_disps)
    
    print(f"Eroded elements:  n={len(eroded_disps):,}, "
          f"mean_disp={eroded_disps.mean():.4f}, std={eroded_disps.std():.4f}")
    print(f"Active elements:  n={len(active_disps):,}, "
          f"mean_disp={active_disps.mean():.4f}, std={active_disps.std():.4f}")
    print(f"Ratio (eroded/active mean): {eroded_disps.mean() / max(active_disps.mean(), 1e-12):.2f}x")
    
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.hist(active_disps, bins=100, alpha=0.6, label=f'Active (n={len(active_disps):,})', 
            density=True, color='steelblue')
    ax.hist(eroded_disps, bins=50, alpha=0.7, label=f'Eroded (n={len(eroded_disps):,})', 
            density=True, color='red')
    ax.set_xlabel('Displacement Magnitude (normalized)')
    ax.set_ylabel('Density')
    ax.set_title('Displacement Distribution: Eroded vs Active Elements')
    ax.legend()
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig('eroded_vs_active_displacement.png', dpi=150, bbox_inches='tight')
    plt.close()
    print("Saved: eroded_vs_active_displacement.png")
else:
    print("No eroded elements found in data")


DISPLACEMENT AT ERODED vs NON-ERODED ELEMENTS
Eroded elements:  n=22,577, mean_disp=0.0190, std=0.0147
Active elements:  n=16,428,156, mean_disp=0.0130, std=0.0140
Ratio (eroded/active mean): 1.47x
Saved: eroded_vs_active_displacement.png


In [8]:
# ============================================================
# CELL 7: Transition analysis — what happens the step BEFORE erosion?
# ============================================================
print("\n" + "=" * 70)
print("PRE-EROSION TRANSITION ANALYSIS")
print("=" * 70)
print("Looking at element state 1 step before erosion occurs...")

pre_erosion_disps = []
pre_erosion_strains = []  # Approximate strain from displacement gradient

for sim_name, sim in all_sims:
    elements = sim[0].elements.cpu().numpy()
    
    for t in range(1, len(sim)):
        if not (hasattr(sim[t], 'x_element') and sim[t].x_element is not None):
            continue
        if not (hasattr(sim[t-1], 'x_element') and sim[t-1].x_element is not None):
            continue
        
        curr_elem = sim[t].x_element.cpu().numpy().flatten()
        prev_elem = sim[t-1].x_element.cpu().numpy().flatten()
        
        # Elements that JUST eroded at step t (active at t-1, eroded at t)
        newly_eroded = (prev_elem >= EROSION_THRESHOLD) & (curr_elem < EROSION_THRESHOLD)
        
        if not newly_eroded.any():
            continue
        
        # Get displacement at t-1 for these elements
        disp_prev = sim[t-1].y.cpu().numpy() if hasattr(sim[t-1], 'y') else sim[t-1].x[:, 2:4].cpu().numpy()
        disp_mag = np.sqrt(disp_prev[:, 0]**2 + disp_prev[:, 1]**2)
        elem_disp = disp_mag[elements].mean(axis=1)
        
        n_new = newly_eroded.sum()
        print(f"  {sim_name} t={t}: {n_new} new erosions, "
              f"pre-erosion disp: mean={elem_disp[newly_eroded].mean():.4f}, "
              f"max={elem_disp[newly_eroded].max():.4f}")
        
        pre_erosion_disps.extend(elem_disp[newly_eroded].tolist())

if pre_erosion_disps:
    pre_erosion_disps = np.array(pre_erosion_disps)
    print(f"\nPre-erosion displacement stats:")
    print(f"  n={len(pre_erosion_disps)}, mean={pre_erosion_disps.mean():.4f}, "
          f"std={pre_erosion_disps.std():.4f}")
    print(f"  min={pre_erosion_disps.min():.4f}, max={pre_erosion_disps.max():.4f}")
    print(f"  p25={np.percentile(pre_erosion_disps, 25):.4f}, "
          f"p50={np.percentile(pre_erosion_disps, 50):.4f}, "
          f"p75={np.percentile(pre_erosion_disps, 75):.4f}")


PRE-EROSION TRANSITION ANALYSIS
Looking at element state 1 step before erosion occurs...
  simulation_179 t=25: 6 new erosions, pre-erosion disp: mean=0.0105, max=0.0121
  simulation_179 t=26: 13 new erosions, pre-erosion disp: mean=0.0103, max=0.0140
  simulation_179 t=27: 9 new erosions, pre-erosion disp: mean=0.0101, max=0.0144
  simulation_179 t=28: 5 new erosions, pre-erosion disp: mean=0.0120, max=0.0133
  simulation_179 t=29: 6 new erosions, pre-erosion disp: mean=0.0143, max=0.0186
  simulation_179 t=30: 3 new erosions, pre-erosion disp: mean=0.0129, max=0.0187
  simulation_179 t=31: 2 new erosions, pre-erosion disp: mean=0.0120, max=0.0164
  simulation_179 t=32: 4 new erosions, pre-erosion disp: mean=0.0105, max=0.0135
  simulation_179 t=33: 1 new erosions, pre-erosion disp: mean=0.0085, max=0.0085
  simulation_179 t=34: 5 new erosions, pre-erosion disp: mean=0.0124, max=0.0174
  simulation_179 t=37: 1 new erosions, pre-erosion disp: mean=0.0254, max=0.0254
  simulation_179 t

In [9]:
# ============================================================
# CELL 8: Summary and recommendations
# ============================================================
print("\n" + "=" * 70)
print("FEASIBILITY SUMMARY")
print("=" * 70)
print(f"""
DATA CHARACTERISTICS:
  - {n_with_erosion}/{len(stats)} simulations have erosion ({100*n_with_erosion/len(stats):.0f}%)
  - x_element is {'BINARY' if len(intermediate) == 0 else 'CONTINUOUS (has intermediate values!)'}
  - Class imbalance: ~{np.mean([s['total_eroded_pct'] for s in stats]):.3f}% of element-timesteps are eroded

APPROACHES (ranked by feasibility):

1. AUXILIARY BINARY HEAD (recommended first step)
   - Add element-level classification head to G-PARCv2
   - Pool node features → element features → MLP → P(erosion)
   - Use focal loss (alpha=0.25, gamma=2) for class imbalance
   - Train jointly with displacement loss
   - Pro: Simple, doesn't change displacement predictions
   - Con: No physical grounding for erosion mechanism

2. THRESHOLD-BASED (zero additional parameters)
   - Compute strain from predicted displacements using MLS
   - Apply physical erosion criterion (e.g., effective plastic strain > threshold)
   - Pro: Physics-based, no training needed
   - Con: Need to know the actual erosion criterion from the FEM solver

3. CONTINUOUS DAMAGE VARIABLE (most ambitious)
   - Requires reprocessing data to include damage evolution
   - Would need damage field from FEM output (not just binary erosion flag)
   - Most physical but needs data that may not be available
""")


FEASIBILITY SUMMARY

DATA CHARACTERISTICS:
  - 24/28 simulations have erosion (86%)
  - x_element is BINARY
  - Class imbalance: ~0.040% of element-timesteps are eroded

APPROACHES (ranked by feasibility):

1. AUXILIARY BINARY HEAD (recommended first step)
   - Add element-level classification head to G-PARCv2
   - Pool node features → element features → MLP → P(erosion)
   - Use focal loss (alpha=0.25, gamma=2) for class imbalance
   - Train jointly with displacement loss
   - Pro: Simple, doesn't change displacement predictions
   - Con: No physical grounding for erosion mechanism

2. THRESHOLD-BASED (zero additional parameters)
   - Compute strain from predicted displacements using MLS
   - Apply physical erosion criterion (e.g., effective plastic strain > threshold)
   - Pro: Physics-based, no training needed
   - Con: Need to know the actual erosion criterion from the FEM solver

3. CONTINUOUS DAMAGE VARIABLE (most ambitious)
   - Requires reprocessing data to include damage evol

In [11]:
# ============================================================
# CELL 9: Von Mises STRAIN at eroded vs non-eroded elements
# ============================================================
# This mirrors Cell 6 but uses MLS gradient solver to compute
# strain from displacement, then compares strain distributions.
# If strain separates better than displacement (1.47x), the
# threshold approach is viable.
# ============================================================

import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath(".")), "G-PARC"))
# Adjust this path if needed:
sys.path.insert(0, "/home/jtb3sud/G-PARC")

import json
from differentiator.hop import SolveGradientsLST

# --- Load normalization stats ---
STATS_FILE = Path("/scratch/jtb3sud/processed_elasto_plastic/global_max/normalized/normalization_stats.json")
# Fallback: try z-score path
# STATS_FILE = Path("/scratch/jtb3sud/processed_elasto_plastic/zscore/normalized/normalization_stats.json")

if STATS_FILE.exists():
    with open(STATS_FILE) as f:
        norm_stats = json.load(f)
    method = norm_stats.get('normalization_method', 'z_score')
    if method == 'global_max':
        norm_method = 'global_max'
        max_position = norm_stats['position']['max_position']
        pos_mean = [0.0, 0.0]
        pos_std = [1.0, 1.0]
    else:
        norm_method = 'z_score'
        max_position = None
        pos_mean = [norm_stats['position']['x_pos']['mean'],
                    norm_stats['position']['y_pos']['mean']]
        pos_std = [norm_stats['position']['x_pos']['std'],
                   norm_stats['position']['y_pos']['std']]
    print(f"Loaded norm stats: method={method}")
else:
    print(f"WARNING: {STATS_FILE} not found, using z-score defaults")
    norm_method = 'z_score'
    max_position = None
    pos_mean = [97.2165, 50.2759]
    pos_std = [59.3803, 28.4965]

# --- Build MLS gradient solver ---
gradient_solver = SolveGradientsLST(
    pos_mean=pos_mean, pos_std=pos_std,
    norm_method=norm_method, max_position=max_position
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

NUM_STATIC = 2   # [x_pos, y_pos]
NUM_DYNAMIC = 2  # [Ux, Uy]
SANITY_LIMIT = 100.0

# --- Compute von Mises strain for each simulation ---
eroded_strains = []
active_strains = []
eroded_disps_check = []   # displacement at eroded elements (sanity check vs Cell 6)
active_disps_check = []

# Also track pre-erosion strain (like Cell 7 but with strain)
pre_erosion_strains = []

for sim_idx, (sim_name, sim) in enumerate(all_sims):
    elements = sim[0].elements.cpu().numpy()
    elements_t = sim[0].elements.to(device)
    num_elements = len(elements)
    
    # Initialize MLS cache for this mesh
    first = sim[0]
    pos0 = first.x[:, :NUM_STATIC].to(device)
    edge0 = first.edge_index.to(device)
    
    # Clear and re-init for each new mesh topology
    if sim_idx == 0 or sim[0].num_nodes != prev_num_nodes:
        gradient_solver.clear_caches()
        dummy = torch.zeros(first.num_nodes, 1, device=device)
        gradient_solver.solve_single_variable(pos0, edge0, dummy)
        print(f"  Initialized MLS for {sim_name} ({first.num_nodes} nodes)")
    
    prev_num_nodes = sim[0].num_nodes
    
    prev_eroded = None
    
    for t, data in enumerate(sim):
        if not (hasattr(data, 'x_element') and data.x_element is not None):
            continue
        
        x_elem = data.x_element.cpu().numpy().flatten()
        eroded = x_elem < EROSION_THRESHOLD
        
        # Get displacement and positions
        pos = data.x[:, :NUM_STATIC].to(device)
        edge_index = data.edge_index.to(device)
        disp = data.x[:, NUM_STATIC:NUM_STATIC + NUM_DYNAMIC].to(device)  # [N, 2]
        
        # Compute strain via MLS
        with torch.no_grad():
            gradients = gradient_solver(
                data.__class__(pos=pos, edge_index=edge_index, num_nodes=pos.shape[0]),
                disp
            )
        
        dUx_dx = torch.clamp(gradients[0][:, 0], -SANITY_LIMIT, SANITY_LIMIT)
        dUx_dy = torch.clamp(gradients[0][:, 1], -SANITY_LIMIT, SANITY_LIMIT)
        dUy_dx = torch.clamp(gradients[1][:, 0], -SANITY_LIMIT, SANITY_LIMIT)
        dUy_dy = torch.clamp(gradients[1][:, 1], -SANITY_LIMIT, SANITY_LIMIT)
        
        eps_xx = dUx_dx
        eps_yy = dUy_dy
        eps_xy = 0.5 * (dUx_dy + dUy_dx)
        
        vm_sq = eps_xx**2 + eps_yy**2 - eps_xx * eps_yy + 3.0 * eps_xy**2
        von_mises = torch.sqrt(torch.clamp(vm_sq, min=1e-10))
        
        # Average to elements
        vm_elem = von_mises[elements_t].mean(dim=1).cpu().numpy()
        
        # Also get displacement magnitude per element for sanity check
        disp_mag = torch.sqrt(disp[:, 0]**2 + disp[:, 1]**2)
        elem_disp = disp_mag[elements_t].mean(dim=1).cpu().numpy()
        
        if eroded.any():
            eroded_strains.extend(vm_elem[eroded].tolist())
            active_strains.extend(vm_elem[~eroded].tolist())
            eroded_disps_check.extend(elem_disp[eroded].tolist())
            active_disps_check.extend(elem_disp[~eroded].tolist())
        
        # Pre-erosion strain (elements about to erode)
        if prev_eroded is not None:
            newly_eroded = (~prev_eroded) & eroded  # was active, now eroded
            if newly_eroded.any():
                pre_erosion_strains.extend(vm_elem[newly_eroded].tolist())
        
        prev_eroded = eroded.copy()

print(f"\nDone computing strain across {len(all_sims)} simulations")

# --- Results ---
eroded_strains = np.array(eroded_strains)
active_strains = np.array(active_strains)
pre_erosion_strains = np.array(pre_erosion_strains)

print(f"\n{'='*70}")
print("VON MISES STRAIN: ERODED vs ACTIVE ELEMENTS")
print(f"{'='*70}")
print(f"Eroded elements:  n={len(eroded_strains):,}, "
      f"mean={eroded_strains.mean():.6f}, std={eroded_strains.std():.6f}")
print(f"Active elements:  n={len(active_strains):,}, "
      f"mean={active_strains.mean():.6f}, std={active_strains.std():.6f}")
print(f"Ratio (eroded/active mean): {eroded_strains.mean() / max(active_strains.mean(), 1e-12):.2f}x")

print(f"\nPercentiles:")
for p in [5, 25, 50, 75, 95]:
    print(f"  p{p}: eroded={np.percentile(eroded_strains, p):.6f}, "
          f"active={np.percentile(active_strains, p):.6f}")

print(f"\nPre-erosion strain (1 step before element erodes):")
print(f"  n={len(pre_erosion_strains)}, mean={pre_erosion_strains.mean():.6f}, "
      f"std={pre_erosion_strains.std():.6f}")
print(f"  min={pre_erosion_strains.min():.6f}, max={pre_erosion_strains.max():.6f}")
for p in [25, 50, 75]:
    print(f"  p{p}={np.percentile(pre_erosion_strains, p):.6f}")

# Sanity check: displacement ratio should match Cell 6 (~1.47x)
if len(eroded_disps_check) > 0:
    ratio_disp = np.mean(eroded_disps_check) / max(np.mean(active_disps_check), 1e-12)
    print(f"\nSanity check — displacement ratio: {ratio_disp:.2f}x (should be ~1.47x)")

# --- Plots ---
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Panel 1: Strain distributions (density)
ax = axes[0]
ax.hist(active_strains, bins=100, alpha=0.6, density=True, 
        color='steelblue', label=f'Active (n={len(active_strains):,})')
ax.hist(eroded_strains, bins=50, alpha=0.7, density=True, 
        color='red', label=f'Eroded (n={len(eroded_strains):,})')
ax.set_xlabel('Von Mises Strain')
ax.set_ylabel('Density')
ax.set_title('Strain Distribution: Eroded vs Active')
ax.legend()
ax.grid(alpha=0.3)

# Panel 2: Pre-erosion strain histogram
ax = axes[1]
ax.hist(pre_erosion_strains, bins=50, alpha=0.7, color='orange', 
        edgecolor='black', label=f'Pre-erosion (n={len(pre_erosion_strains)})')
ax.axvline(pre_erosion_strains.mean(), color='red', ls='--', 
           label=f'Mean={pre_erosion_strains.mean():.5f}')
ax.axvline(np.median(pre_erosion_strains), color='darkred', ls=':', 
           label=f'Median={np.median(pre_erosion_strains):.5f}')
ax.set_xlabel('Von Mises Strain')
ax.set_ylabel('Count')
ax.set_title('Strain at Elements 1-Step Before Erosion')
ax.legend(fontsize=8)
ax.grid(alpha=0.3)

# Panel 3: Quick threshold sweep (F1 vs threshold)
ax = axes[2]
# Combine all element-timestep data for sweep
all_vm = np.concatenate([eroded_strains, active_strains])
all_gt = np.concatenate([np.ones(len(eroded_strains), dtype=bool),
                         np.zeros(len(active_strains), dtype=bool)])
# Subsample if huge (>1M)
if len(all_vm) > 1_000_000:
    idx = np.random.choice(len(all_vm), 1_000_000, replace=False)
    all_vm_sub, all_gt_sub = all_vm[idx], all_gt[idx]
else:
    all_vm_sub, all_gt_sub = all_vm, all_gt

from sklearn.metrics import f1_score, precision_score, recall_score

thresholds = np.linspace(
    np.percentile(pre_erosion_strains, 5) * 0.5 if len(pre_erosion_strains) > 0 else 0,
    np.percentile(pre_erosion_strains, 95) * 2.0 if len(pre_erosion_strains) > 0 else all_vm.max(),
    100
)
f1s, precs, recs = [], [], []
for thr in thresholds:
    pred = all_vm_sub > thr
    f1s.append(f1_score(all_gt_sub, pred, zero_division=0))
    precs.append(precision_score(all_gt_sub, pred, zero_division=0))
    recs.append(recall_score(all_gt_sub, pred, zero_division=0))

ax.plot(thresholds, f1s, 'b-', lw=2, label='F1')
ax.plot(thresholds, precs, 'g--', lw=1.5, label='Precision')
ax.plot(thresholds, recs, 'r--', lw=1.5, label='Recall')
best_thr = thresholds[np.argmax(f1s)]
ax.axvline(best_thr, color='k', ls=':', label=f'Best thr={best_thr:.5f} (F1={max(f1s):.3f})')
ax.set_xlabel('Von Mises Strain Threshold')
ax.set_ylabel('Score')
ax.set_title('Threshold Sweep: Can Strain Predict Erosion?')
ax.legend(fontsize=8)
ax.grid(alpha=0.3)
ax.set_ylim(-0.05, 1.05)

plt.suptitle(f'Von Mises Strain Feasibility — Separation ratio: '
             f'{eroded_strains.mean()/max(active_strains.mean(),1e-12):.2f}x '
             f'(vs 1.47x for displacement)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('strain_vs_erosion_feasibility.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: strain_vs_erosion_feasibility.png")

print(f"\n{'='*70}")
print("VERDICT")
print(f"{'='*70}")
ratio = eroded_strains.mean() / max(active_strains.mean(), 1e-12)
if ratio > 3.0:
    print(f"  Strain separation: {ratio:.1f}x → GOOD. Threshold approach is viable.")
elif ratio > 2.0:
    print(f"  Strain separation: {ratio:.1f}x → MODERATE. Threshold may work with tuning.")
else:
    print(f"  Strain separation: {ratio:.1f}x → WEAK (similar to displacement at 1.47x).")
    print(f"  Consider learned approach instead (auxiliary head or two-stage).")
print(f"  Best F1 from threshold sweep: {max(f1s):.3f}")

Loaded norm stats: method=global_max
Device: cpu
  Initialized MLS for simulation_179 (24670 nodes)
  Initialized MLS for simulation_191 (27424 nodes)
  Initialized MLS for simulation_337 (25429 nodes)
  Initialized MLS for simulation_384 (24322 nodes)
  Initialized MLS for simulation_395 (25907 nodes)
  Initialized MLS for simulation_490 (25008 nodes)
  Initialized MLS for simulation_676 (26917 nodes)
  Initialized MLS for simulation_989 (24869 nodes)
  Initialized MLS for simulation_106 (25342 nodes)
  Initialized MLS for simulation_127 (24645 nodes)
  Initialized MLS for simulation_13 (23491 nodes)
  Initialized MLS for simulation_133 (24915 nodes)
  Initialized MLS for simulation_135 (29505 nodes)
  Initialized MLS for simulation_15 (24893 nodes)
  Initialized MLS for simulation_202 (26255 nodes)
  Initialized MLS for simulation_207 (24954 nodes)
  Initialized MLS for simulation_208 (25901 nodes)
  Initialized MLS for simulation_214 (23365 nodes)
  Initialized MLS for simulation_22

In [12]:
# ============================================================
# CELL 10: Test threshold on GT displacement — visual check
# ============================================================
# Pick a threshold and see how well it predicts erosion
# spatially on a few simulations using ground truth displacement.
# ============================================================

from matplotlib.collections import PolyCollection
from matplotlib.colors import Normalize

# Use the best threshold from Cell 9's sweep
# (or manually set one near the pre-erosion p25 ~0.70)
THRESHOLD = best_thr
print(f"Using threshold: {THRESHOLD:.5f}")

# Pick sims that have erosion
eroding_sims_list = [(name, sim) for name, sim in all_sims 
                     if any(hasattr(d, 'x_element') and d.x_element is not None
                            and (d.x_element.cpu().numpy().flatten() < EROSION_THRESHOLD).any()
                            for d in sim)]

NUM_SIMS_TO_SHOW = min(3, len(eroding_sims_list))

for viz_idx in range(NUM_SIMS_TO_SHOW):
    sim_name, sim = eroding_sims_list[viz_idx]
    
    elements = sim[0].elements.cpu().numpy()
    elements_t = sim[0].elements.to(device)
    num_elements = len(elements)
    pos_ref = sim[0].x[:, :NUM_STATIC].cpu().numpy()
    
    # Recompute MLS if needed
    pos0 = sim[0].x[:, :NUM_STATIC].to(device)
    edge0 = sim[0].edge_index.to(device)
    gradient_solver.clear_caches()
    dummy = torch.zeros(sim[0].num_nodes, 1, device=device)
    gradient_solver.solve_single_variable(pos0, edge0, dummy)
    
    # Find timesteps to visualize:
    # first erosion, middle of erosion, and final timestep
    first_erosion_t = None
    for t, data in enumerate(sim):
        if hasattr(data, 'x_element') and data.x_element is not None:
            if (data.x_element.cpu().numpy().flatten() < EROSION_THRESHOLD).any():
                first_erosion_t = t
                break
    
    if first_erosion_t is None:
        continue
    
    last_t = len(sim) - 1
    mid_t = (first_erosion_t + last_t) // 2
    viz_timesteps = [first_erosion_t, mid_t, last_t]
    
    # Compute predictions at each timestep
    fig, axes = plt.subplots(3, 3, figsize=(18, 16))
    
    for col, t in enumerate(viz_timesteps):
        data = sim[t]
        
        # GT erosion
        x_elem = data.x_element.cpu().numpy().flatten()
        gt_eroded = x_elem < EROSION_THRESHOLD
        
        # Compute strain from GT displacement
        pos = data.x[:, :NUM_STATIC].to(device)
        edge_index = data.edge_index.to(device)
        disp = data.x[:, NUM_STATIC:NUM_STATIC + NUM_DYNAMIC].to(device)
        
        with torch.no_grad():
            from torch_geometric.data import Data as PyGData
            mesh_data = PyGData(pos=pos, edge_index=edge_index, num_nodes=pos.shape[0])
            gradients = gradient_solver(mesh_data, disp)
        
        dUx_dx = torch.clamp(gradients[0][:, 0], -SANITY_LIMIT, SANITY_LIMIT)
        dUx_dy = torch.clamp(gradients[0][:, 1], -SANITY_LIMIT, SANITY_LIMIT)
        dUy_dx = torch.clamp(gradients[1][:, 0], -SANITY_LIMIT, SANITY_LIMIT)
        dUy_dy = torch.clamp(gradients[1][:, 1], -SANITY_LIMIT, SANITY_LIMIT)
        
        eps_xx = dUx_dx
        eps_yy = dUy_dy
        eps_xy = 0.5 * (dUx_dy + dUy_dx)
        vm_sq = eps_xx**2 + eps_yy**2 - eps_xx * eps_yy + 3.0 * eps_xy**2
        von_mises = torch.sqrt(torch.clamp(vm_sq, min=1e-10))
        
        vm_elem = von_mises[elements_t].mean(dim=1).cpu().numpy()
        pred_eroded = vm_elem > THRESHOLD
        
        # Metrics for this timestep
        tp = (pred_eroded & gt_eroded).sum()
        fp = (pred_eroded & ~gt_eroded).sum()
        fn = (~pred_eroded & gt_eroded).sum()
        prec = tp / max(tp + fp, 1)
        rec = tp / max(tp + fn, 1)
        f1 = 2 * prec * rec / max(prec + rec, 1e-12)
        
        # --- Row 1: Ground truth erosion ---
        ax = axes[0, col]
        poly_verts = pos_ref[elements]
        valid_verts = poly_verts[~gt_eroded]
        eroded_verts = poly_verts[gt_eroded]
        
        if len(valid_verts) > 0:
            pc = PolyCollection(valid_verts, facecolors='lightblue',
                                edgecolors='k', linewidths=0.05)
            ax.add_collection(pc)
        if len(eroded_verts) > 0:
            pc = PolyCollection(eroded_verts, facecolors='red',
                                edgecolors='darkred', linewidths=0.1, alpha=0.8)
            ax.add_collection(pc)
        
        ax.set_xlim(pos_ref[:, 0].min(), pos_ref[:, 0].max())
        ax.set_ylim(pos_ref[:, 1].min(), pos_ref[:, 1].max())
        ax.set_aspect('equal')
        ax.set_title(f't={t} — GT ({gt_eroded.sum()} eroded)', fontsize=10)
        ax.axis('off')
        if col == 0:
            ax.set_ylabel('Ground Truth', fontsize=12, rotation=0, labelpad=80, va='center')
        
        # --- Row 2: Predicted erosion ---
        ax = axes[1, col]
        valid_verts_p = poly_verts[~pred_eroded]
        eroded_verts_p = poly_verts[pred_eroded]
        
        if len(valid_verts_p) > 0:
            pc = PolyCollection(valid_verts_p, facecolors='lightblue',
                                edgecolors='k', linewidths=0.05)
            ax.add_collection(pc)
        if len(eroded_verts_p) > 0:
            pc = PolyCollection(eroded_verts_p, facecolors='red',
                                edgecolors='darkred', linewidths=0.1, alpha=0.8)
            ax.add_collection(pc)
        
        ax.set_xlim(pos_ref[:, 0].min(), pos_ref[:, 0].max())
        ax.set_ylim(pos_ref[:, 1].min(), pos_ref[:, 1].max())
        ax.set_aspect('equal')
        ax.set_title(f't={t} — Pred ({pred_eroded.sum()} eroded)\n'
                     f'P={prec:.2f} R={rec:.2f} F1={f1:.2f}', fontsize=10)
        ax.axis('off')
        if col == 0:
            ax.set_ylabel('Predicted', fontsize=12, rotation=0, labelpad=80, va='center')
        
        # --- Row 3: Von Mises strain field ---
        ax = axes[2, col]
        vmax = np.percentile(vm_elem[~gt_eroded], 99) if (~gt_eroded).sum() > 0 else vm_elem.max()
        vmax = max(vmax, THRESHOLD * 1.5)
        norm = Normalize(vmin=0, vmax=vmax)
        cmap = plt.colormaps.get_cmap('hot')
        
        colors = cmap(norm(np.clip(vm_elem, 0, vmax)))
        pc = PolyCollection(poly_verts, facecolors=colors, edgecolors='k', linewidths=0.02)
        ax.add_collection(pc)
        
        ax.set_xlim(pos_ref[:, 0].min(), pos_ref[:, 0].max())
        ax.set_ylim(pos_ref[:, 1].min(), pos_ref[:, 1].max())
        ax.set_aspect('equal')
        ax.set_title(f't={t} — Von Mises Strain', fontsize=10)
        ax.axis('off')
        if col == 0:
            ax.set_ylabel('Strain Field', fontsize=12, rotation=0, labelpad=80, va='center')
        
        # Add colorbar
        sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
        sm.set_array([])
        plt.colorbar(sm, ax=ax, fraction=0.046, pad=0.04)
    
    fig.suptitle(f'{sim_name} — Erosion Prediction from GT Strain '
                 f'(threshold={THRESHOLD:.4f})',
                 fontsize=14, fontweight='bold')
    plt.tight_layout()
    fname = f'erosion_threshold_test_{sim_name}.png'
    plt.savefig(fname, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Saved: {fname}")

# --- Aggregate metrics across ALL eroding sims, ALL timesteps ---
print(f"\n{'='*70}")
print(f"AGGREGATE METRICS (threshold={THRESHOLD:.5f})")
print(f"{'='*70}")

total_tp, total_fp, total_fn = 0, 0, 0
n_timesteps_with_erosion = 0

for sim_name, sim in eroding_sims_list:
    elements_t = sim[0].elements.to(device)
    elements_np = sim[0].elements.cpu().numpy()
    
    gradient_solver.clear_caches()
    pos0 = sim[0].x[:, :NUM_STATIC].to(device)
    edge0 = sim[0].edge_index.to(device)
    dummy = torch.zeros(sim[0].num_nodes, 1, device=device)
    gradient_solver.solve_single_variable(pos0, edge0, dummy)
    
    for t, data in enumerate(sim):
        if not (hasattr(data, 'x_element') and data.x_element is not None):
            continue
        
        x_elem = data.x_element.cpu().numpy().flatten()
        gt_eroded = x_elem < EROSION_THRESHOLD
        
        if not gt_eroded.any():
            continue
        
        n_timesteps_with_erosion += 1
        
        pos = data.x[:, :NUM_STATIC].to(device)
        edge_index = data.edge_index.to(device)
        disp = data.x[:, NUM_STATIC:NUM_STATIC + NUM_DYNAMIC].to(device)
        
        with torch.no_grad():
            mesh_data = PyGData(pos=pos, edge_index=edge_index, num_nodes=pos.shape[0])
            gradients = gradient_solver(mesh_data, disp)
        
        dUx_dx = torch.clamp(gradients[0][:, 0], -SANITY_LIMIT, SANITY_LIMIT)
        dUx_dy = torch.clamp(gradients[0][:, 1], -SANITY_LIMIT, SANITY_LIMIT)
        dUy_dx = torch.clamp(gradients[1][:, 0], -SANITY_LIMIT, SANITY_LIMIT)
        dUy_dy = torch.clamp(gradients[1][:, 1], -SANITY_LIMIT, SANITY_LIMIT)
        
        eps_xx = dUx_dx
        eps_yy = dUy_dy
        eps_xy = 0.5 * (dUx_dy + dUy_dx)
        vm_sq = eps_xx**2 + eps_yy**2 - eps_xx * eps_yy + 3.0 * eps_xy**2
        von_mises = torch.sqrt(torch.clamp(vm_sq, min=1e-10))
        
        vm_elem = von_mises[elements_t].mean(dim=1).cpu().numpy()
        pred_eroded = vm_elem > THRESHOLD
        
        total_tp += (pred_eroded & gt_eroded).sum()
        total_fp += (pred_eroded & ~gt_eroded).sum()
        total_fn += (~pred_eroded & gt_eroded).sum()

global_prec = total_tp / max(total_tp + total_fp, 1)
global_rec = total_tp / max(total_tp + total_fn, 1)
global_f1 = 2 * global_prec * global_rec / max(global_prec + global_rec, 1e-12)

print(f"Timesteps with erosion: {n_timesteps_with_erosion}")
print(f"Precision: {global_prec:.4f}")
print(f"Recall:    {global_rec:.4f}")
print(f"F1:        {global_f1:.4f}")
print(f"TP={total_tp}, FP={total_fp}, FN={total_fn}")

Using threshold: 0.83999
Saved: erosion_threshold_test_simulation_179.png
Saved: erosion_threshold_test_simulation_191.png
Saved: erosion_threshold_test_simulation_337.png

AGGREGATE METRICS (threshold=0.83999)
Timesteps with erosion: 327
Precision: 0.5804
Recall:    0.8647
F1:        0.6946
TP=19523, FP=14117, FN=3054


In [15]:
# ============================================================
# CELL 11: Erosion from PREDICTED displacement (model rollout)
# ============================================================
# This is the real test — same threshold approach but using
# the model's predicted displacement instead of GT.
# Compares to Cell 10 results to see how much model error hurts.
# ============================================================

import json
from pathlib import Path
from torch_geometric.data import Data as PyGData

sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath(".")), "G-PARC"))
# Adjust if needed:
# sys.path.insert(0, "/home/jtb3sud/G-PARC")

from utilities.featureextractor import GraphConvFeatureExtractorV2
from differentiator.differentiator import ElastoPlasticDifferentiator
from differentiator.hop import SolveGradientsLST, SolveWeightLST2d
from models.globalelasto import GPARC_ElastoPlastic_Numerical

# =============================================
# CONFIG — matches 2hop SLURM training script
# =============================================
CHECKPOINT_PATH = "/scratch/jtb3sud/elasto_graphconv_V2/2hop/best_model.pth"

# Architecture args (from 2hop SLURM script)
HIDDEN_CHANNELS = 128
FEATURE_OUT_CHANNELS = 128
NUM_LAYERS = 4
DROPOUT = 0.0
SPADE_HEADS = 4
SPADE_CONCAT = True           # --spade_concat in training
SPADE_DROPOUT = 0.1
USE_VON_MISES = True          # --use_von_mises
USE_VOLUMETRIC = True         # --use_volumetric
N_STATE_VAR = 0
ZERO_INIT = True              # --zero_init
INTEGRATOR = "euler"
CLAMP_OUTPUT = False           # --no_clamp_output in training
USE_LAYER_NORM = True          # --use_layer_norm
USE_RELATIVE_POS = True        # --use_relative_pos

# Threshold from Cell 9
# THRESHOLD = best_thr       # use this if Cell 9 was run in same session
THRESHOLD = 0.84              # from Cell 9 result

NUM_SIMS = 3                  # how many test sims to evaluate
# =============================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# --- Load checkpoint ---
checkpoint = torch.load(CHECKPOINT_PATH, map_location=device, weights_only=False)
print(f"Loaded checkpoint: epoch {checkpoint.get('epoch', '?')}, "
      f"val_loss={checkpoint.get('val_loss', '?')}")

# --- Normalization stats ---
# (reuse from Cell 9 if available, otherwise reload)
try:
    _ = norm_method
    print(f"Reusing norm stats from Cell 9: method={norm_method}")
except NameError:
    STATS_FILE = Path("/scratch/jtb3sud/processed_elasto_plastic/global_max/normalized/normalization_stats.json")
    with open(STATS_FILE) as f:
        norm_stats = json.load(f)
    method = norm_stats.get('normalization_method', 'z_score')
    if method == 'global_max':
        norm_method = 'global_max'
        max_position = norm_stats['position']['max_position']
        pos_mean = [0.0, 0.0]
        pos_std = [1.0, 1.0]
    else:
        norm_method = 'z_score'
        max_position = None
        pos_mean = [norm_stats['position']['x_pos']['mean'],
                    norm_stats['position']['y_pos']['mean']]
        pos_std = [norm_stats['position']['x_pos']['std'],
                   norm_stats['position']['y_pos']['std']]

# --- Build model ---
gradient_solver_model = SolveGradientsLST(
    pos_mean=pos_mean, pos_std=pos_std,
    norm_method=norm_method, max_position=max_position
)
laplacian_solver = SolveWeightLST2d(
    pos_mean=pos_mean, pos_std=pos_std,
    norm_method=norm_method, max_position=max_position,
    use_2hop_extension=True
)

feature_extractor = GraphConvFeatureExtractorV2(
    in_channels=NUM_STATIC,
    hidden_channels=HIDDEN_CHANNELS,
    out_channels=FEATURE_OUT_CHANNELS,
    num_layers=NUM_LAYERS,
    dropout=DROPOUT,
    use_layer_norm=USE_LAYER_NORM,
    use_relative_pos=USE_RELATIVE_POS,
)

derivative_solver = ElastoPlasticDifferentiator(
    num_static_feats=NUM_STATIC,
    num_dynamic_feats=NUM_DYNAMIC,
    feature_extractor=feature_extractor,
    gradient_solver=gradient_solver_model,
    laplacian_solver=laplacian_solver,
    n_fe_features=FEATURE_OUT_CHANNELS,
    list_strain_idx=list(range(NUM_DYNAMIC)),
    list_laplacian_idx=list(range(NUM_DYNAMIC)),
    spade_random_noise=False,
    heads=SPADE_HEADS,
    concat=SPADE_CONCAT,
    dropout=SPADE_DROPOUT,
    use_von_mises=USE_VON_MISES,
    use_volumetric=USE_VOLUMETRIC,
    n_state_var=N_STATE_VAR,
    zero_init=ZERO_INIT,
)

model = GPARC_ElastoPlastic_Numerical(
    derivative_solver_physics=derivative_solver,
    integrator_type=INTEGRATOR,
    num_static_feats=NUM_STATIC,
    num_dynamic_feats=NUM_DYNAMIC,
    pos_mean=pos_mean,
    pos_std=pos_std,
    boundary_threshold=0.5,
    clamp_output=CLAMP_OUTPUT,
    norm_method=norm_method,
    max_position=max_position,
)

model.load_state_dict(checkpoint['model_state_dict'])
model.to(device)
model.eval()
print("Model loaded and ready")

# --- Separate gradient solver for post-hoc strain computation ---
# (the model's internal solver may have different cache state)
gradient_solver_eval = SolveGradientsLST(
    pos_mean=pos_mean, pos_std=pos_std,
    norm_method=norm_method, max_position=max_position
)

# --- Run rollout + erosion prediction ---
eroding_test = [(n, s) for n, s in test_sims
                if any(hasattr(d, 'x_element') and d.x_element is not None
                       and (d.x_element.cpu().numpy().flatten() < EROSION_THRESHOLD).any()
                       for d in s)]
print(f"\nTest sims with erosion: {len(eroding_test)}")

sims_to_eval = eroding_test[:NUM_SIMS]

total_tp_pred, total_fp_pred, total_fn_pred = 0, 0, 0
total_tp_gt, total_fp_gt, total_fn_gt = 0, 0, 0
n_eval_steps = 0

for viz_idx, (sim_name, sim) in enumerate(sims_to_eval):
    print(f"\n--- {sim_name} ({len(sim)} timesteps) ---")
    
    elements_np = sim[0].elements.cpu().numpy()
    elements_t = sim[0].elements.to(device)
    num_elements = len(elements_np)
    pos_ref = sim[0].x[:, :NUM_STATIC].cpu().numpy()
    
    # Rollout
    num_steps = len(sim) - 1
    with torch.no_grad():
        states = model.rollout(sim, num_steps, device)
    print(f"  Rollout: {len(states)} states")
    
    # Init eval gradient solver for this mesh
    gradient_solver_eval.clear_caches()
    pos0 = sim[0].x[:, :NUM_STATIC].to(device)
    edge0 = sim[0].edge_index.to(device)
    dummy = torch.zeros(sim[0].num_nodes, 1, device=device)
    gradient_solver_eval.solve_single_variable(pos0, edge0, dummy)
    
    # Find key timesteps for visualization
    first_erosion_t = None
    for t in range(len(sim)):
        if hasattr(sim[t], 'x_element') and sim[t].x_element is not None:
            if (sim[t].x_element.cpu().numpy().flatten() < EROSION_THRESHOLD).any():
                first_erosion_t = t
                break
    
    last_t = min(len(states) - 1, len(sim) - 1)
    mid_t = (first_erosion_t + last_t) // 2 if first_erosion_t else last_t // 2
    viz_timesteps = [first_erosion_t or 0, mid_t, last_t]
    
    # --- Visualization: 3 rows x 3 cols per timestep ---
    # Row 1: GT erosion
    # Row 2: Erosion from PREDICTED displacement  
    # Row 3: Erosion from GT displacement (for comparison)
    fig, axes = plt.subplots(3, 3, figsize=(18, 16))
    poly_verts = pos_ref[elements_np]
    
    for col, t in enumerate(viz_timesteps):
        if t >= len(states) or t >= len(sim):
            continue
        
        data_t = sim[t]
        if not (hasattr(data_t, 'x_element') and data_t.x_element is not None):
            continue
        
        # GT erosion
        gt_eroded = data_t.x_element.cpu().numpy().flatten() < EROSION_THRESHOLD
        
        pos = data_t.x[:, :NUM_STATIC].to(device)
        edge_index = data_t.edge_index.to(device)
        
        # --- Strain from PREDICTED displacement ---
        pred_disp = torch.tensor(states[t], dtype=torch.float32, device=device)
        
        with torch.no_grad():
            mesh_data = PyGData(pos=pos, edge_index=edge_index, num_nodes=pos.shape[0])
            grads_pred = gradient_solver_eval(mesh_data, pred_disp)
        
        dUx_dx = torch.clamp(grads_pred[0][:, 0], -SANITY_LIMIT, SANITY_LIMIT)
        dUx_dy = torch.clamp(grads_pred[0][:, 1], -SANITY_LIMIT, SANITY_LIMIT)
        dUy_dx = torch.clamp(grads_pred[1][:, 0], -SANITY_LIMIT, SANITY_LIMIT)
        dUy_dy = torch.clamp(grads_pred[1][:, 1], -SANITY_LIMIT, SANITY_LIMIT)
        eps_xx = dUx_dx; eps_yy = dUy_dy
        eps_xy = 0.5 * (dUx_dy + dUy_dx)
        vm_sq = eps_xx**2 + eps_yy**2 - eps_xx * eps_yy + 3.0 * eps_xy**2
        vm_pred = torch.sqrt(torch.clamp(vm_sq, min=1e-10))
        vm_pred_elem = vm_pred[elements_t].mean(dim=1).cpu().numpy()
        pred_eroded_from_model = vm_pred_elem > THRESHOLD
        
        # --- Strain from GT displacement ---
        gt_disp = data_t.x[:, NUM_STATIC:NUM_STATIC + NUM_DYNAMIC].to(device)
        
        with torch.no_grad():
            grads_gt = gradient_solver_eval(mesh_data, gt_disp)
        
        dUx_dx = torch.clamp(grads_gt[0][:, 0], -SANITY_LIMIT, SANITY_LIMIT)
        dUx_dy = torch.clamp(grads_gt[0][:, 1], -SANITY_LIMIT, SANITY_LIMIT)
        dUy_dx = torch.clamp(grads_gt[1][:, 0], -SANITY_LIMIT, SANITY_LIMIT)
        dUy_dy = torch.clamp(grads_gt[1][:, 1], -SANITY_LIMIT, SANITY_LIMIT)
        eps_xx = dUx_dx; eps_yy = dUy_dy
        eps_xy = 0.5 * (dUx_dy + dUy_dx)
        vm_sq = eps_xx**2 + eps_yy**2 - eps_xx * eps_yy + 3.0 * eps_xy**2
        vm_gt = torch.sqrt(torch.clamp(vm_sq, min=1e-10))
        vm_gt_elem = vm_gt[elements_t].mean(dim=1).cpu().numpy()
        pred_eroded_from_gt = vm_gt_elem > THRESHOLD
        
        # Metrics
        tp_p = (pred_eroded_from_model & gt_eroded).sum()
        fp_p = (pred_eroded_from_model & ~gt_eroded).sum()
        fn_p = (~pred_eroded_from_model & gt_eroded).sum()
        prec_p = tp_p / max(tp_p + fp_p, 1)
        rec_p = tp_p / max(tp_p + fn_p, 1)
        f1_p = 2 * prec_p * rec_p / max(prec_p + rec_p, 1e-12)
        
        tp_g = (pred_eroded_from_gt & gt_eroded).sum()
        fp_g = (pred_eroded_from_gt & ~gt_eroded).sum()
        fn_g = (~pred_eroded_from_gt & gt_eroded).sum()
        prec_g = tp_g / max(tp_g + fp_g, 1)
        rec_g = tp_g / max(tp_g + fn_g, 1)
        f1_g = 2 * prec_g * rec_g / max(prec_g + rec_g, 1e-12)
        
        if gt_eroded.any():
            total_tp_pred += tp_p; total_fp_pred += fp_p; total_fn_pred += fn_p
            total_tp_gt += tp_g; total_fp_gt += fp_g; total_fn_gt += fn_g
            n_eval_steps += 1
        
        # --- ROW 1: Ground truth erosion ---
        ax = axes[0, col]
        valid_v = poly_verts[~gt_eroded]
        eroded_v = poly_verts[gt_eroded]
        if len(valid_v) > 0:
            ax.add_collection(PolyCollection(valid_v, facecolors='lightblue',
                                             edgecolors='k', linewidths=0.05))
        if len(eroded_v) > 0:
            ax.add_collection(PolyCollection(eroded_v, facecolors='red',
                                             edgecolors='darkred', linewidths=0.1, alpha=0.8))
        ax.set_xlim(pos_ref[:, 0].min(), pos_ref[:, 0].max())
        ax.set_ylim(pos_ref[:, 1].min(), pos_ref[:, 1].max())
        ax.set_aspect('equal'); ax.axis('off')
        ax.set_title(f't={t} — GT ({gt_eroded.sum()} eroded)', fontsize=10)
        if col == 0:
            ax.set_ylabel('Ground Truth', fontsize=11, rotation=0, labelpad=90, va='center')
        
        # --- ROW 2: Erosion from PREDICTED displacement ---
        ax = axes[1, col]
        valid_v = poly_verts[~pred_eroded_from_model]
        eroded_v = poly_verts[pred_eroded_from_model]
        if len(valid_v) > 0:
            ax.add_collection(PolyCollection(valid_v, facecolors='lightblue',
                                             edgecolors='k', linewidths=0.05))
        if len(eroded_v) > 0:
            ax.add_collection(PolyCollection(eroded_v, facecolors='red',
                                             edgecolors='darkred', linewidths=0.1, alpha=0.8))
        ax.set_xlim(pos_ref[:, 0].min(), pos_ref[:, 0].max())
        ax.set_ylim(pos_ref[:, 1].min(), pos_ref[:, 1].max())
        ax.set_aspect('equal'); ax.axis('off')
        ax.set_title(f't={t} — From Model Disp ({pred_eroded_from_model.sum()} pred)\n'
                     f'P={prec_p:.2f} R={rec_p:.2f} F1={f1_p:.2f}', fontsize=10)
        if col == 0:
            ax.set_ylabel('From Predicted\nDisplacement', fontsize=11, rotation=0,
                          labelpad=90, va='center')
        
        # --- ROW 3: Erosion from GT displacement (reference) ---
        ax = axes[2, col]
        valid_v = poly_verts[~pred_eroded_from_gt]
        eroded_v = poly_verts[pred_eroded_from_gt]
        if len(valid_v) > 0:
            ax.add_collection(PolyCollection(valid_v, facecolors='lightblue',
                                             edgecolors='k', linewidths=0.05))
        if len(eroded_v) > 0:
            ax.add_collection(PolyCollection(eroded_v, facecolors='red',
                                             edgecolors='darkred', linewidths=0.1, alpha=0.8))
        ax.set_xlim(pos_ref[:, 0].min(), pos_ref[:, 0].max())
        ax.set_ylim(pos_ref[:, 1].min(), pos_ref[:, 1].max())
        ax.set_aspect('equal'); ax.axis('off')
        ax.set_title(f't={t} — From GT Disp ({pred_eroded_from_gt.sum()} pred)\n'
                     f'P={prec_g:.2f} R={rec_g:.2f} F1={f1_g:.2f}', fontsize=10)
        if col == 0:
            ax.set_ylabel('From GT\nDisplacement', fontsize=11, rotation=0,
                          labelpad=90, va='center')
    
    fig.suptitle(f'{sim_name} — Erosion: Model vs GT Displacement '
                 f'(threshold={THRESHOLD:.4f})', fontsize=13, fontweight='bold')
    plt.tight_layout()
    fname = f'erosion_model_vs_gt_{sim_name}.png'
    plt.savefig(fname, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"  Saved: {fname}")

# --- Aggregate ---
print(f"\n{'='*70}")
print(f"AGGREGATE: MODEL PREDICTED vs GT DISPLACEMENT (threshold={THRESHOLD:.5f})")
print(f"{'='*70}")
print(f"Timesteps evaluated: {n_eval_steps}")

if n_eval_steps > 0:
    gp = total_tp_pred / max(total_tp_pred + total_fp_pred, 1)
    gr = total_tp_pred / max(total_tp_pred + total_fn_pred, 1)
    gf = 2 * gp * gr / max(gp + gr, 1e-12)
    
    gp_gt = total_tp_gt / max(total_tp_gt + total_fp_gt, 1)
    gr_gt = total_tp_gt / max(total_tp_gt + total_fn_gt, 1)
    gf_gt = 2 * gp_gt * gr_gt / max(gp_gt + gr_gt, 1e-12)
    
    print(f"\nFrom MODEL displacement:")
    print(f"  Precision: {gp:.4f}  Recall: {gr:.4f}  F1: {gf:.4f}")
    print(f"  TP={total_tp_pred}, FP={total_fp_pred}, FN={total_fn_pred}")
    
    print(f"\nFrom GT displacement (Cell 10 reference):")
    print(f"  Precision: {gp_gt:.4f}  Recall: {gr_gt:.4f}  F1: {gf_gt:.4f}")
    print(f"  TP={total_tp_gt}, FP={total_fp_gt}, FN={total_fn_gt}")
    
    print(f"\nDegradation:")
    print(f"  F1:        {gf_gt:.4f} → {gf:.4f}  ({(gf/max(gf_gt,1e-12)-1)*100:+.1f}%)")
    print(f"  Precision: {gp_gt:.4f} → {gp:.4f}")
    print(f"  Recall:    {gr_gt:.4f} → {gr:.4f}")

Device: cpu
Loaded checkpoint: epoch 1039, val_loss=?
Reusing norm stats from Cell 9: method=global_max
Model loaded and ready

Test sims with erosion: 8

--- simulation_179 (40 timesteps) ---
Initializing MLS operator weights...
  2-hop stencil: added 5129 edges (20.8% of nodes extended)
✓ MLS weights initialized
  2-hop stencil: added 5129 edges (20.8% of nodes extended)
  2-hop stencil: added 5129 edges (20.8% of nodes extended)
  2-hop stencil: added 5129 edges (20.8% of nodes extended)
  2-hop stencil: added 5129 edges (20.8% of nodes extended)
  2-hop stencil: added 5129 edges (20.8% of nodes extended)
  2-hop stencil: added 5129 edges (20.8% of nodes extended)
  2-hop stencil: added 5129 edges (20.8% of nodes extended)
  2-hop stencil: added 5129 edges (20.8% of nodes extended)
  2-hop stencil: added 5129 edges (20.8% of nodes extended)
  2-hop stencil: added 5129 edges (20.8% of nodes extended)
  2-hop stencil: added 5129 edges (20.8% of nodes extended)
  2-hop stencil: added 5